In [3]:
import polars as pl

In [4]:
yelp = pl.read_parquet("~/shared/ling-583/yelp.parquet")

In [5]:
yelp.columns

['categories',
 'review_id',
 'stars',
 'date',
 'text',
 'business_id',
 'name',
 'city',
 'state',
 'lang']

In [4]:
rest_counts = (
    yelp.group_by(["name", "city"])
        .len()
        .sort("len", descending=True)
)

rest_counts.head(20)

name,city,len
str,str,u32
"""Hash House A Go Go""","""Las Vegas""",6244
"""Mon Ami Gabi""","""Las Vegas""",5558
"""Earl of Sandwich""","""Las Vegas""",4408
"""Gordon Ramsay BurGR""","""Las Vegas""",4149
"""Serendipity 3""","""Las Vegas""",3330
…,…,…
"""Ellis Island Casino & Brewery""","""Las Vegas""",1551
"""Bobby Q""","""Phoenix""",1446
"""Grimaldi's Pizzeria""","""Las Vegas""",1432


In [7]:
restaurant_name = "Hash House A Go Go"
restaurant_city = "Las Vegas"

rest_reviews = yelp.filter(
    (pl.col("name") == restaurant_name) &
    (pl.col("city") == restaurant_city)
)

len(rest_reviews)

6244

In [9]:
yelp.filter(
    (pl.col("name") == "Hash House A Go Go") &
    (pl.col("city") == "Las Vegas")
).select("categories")

categories
list[str]
"[""Breakfast & Brunch"", ""American (New)"", ""Restaurants""]"
"[""Breakfast & Brunch"", ""American (New)"", ""Restaurants""]"
"[""Breakfast & Brunch"", ""American (New)"", ""Restaurants""]"
"[""Breakfast & Brunch"", ""American (New)"", ""Restaurants""]"
"[""Breakfast & Brunch"", ""American (New)"", ""Restaurants""]"
…
"[""Breakfast & Brunch"", ""American (New)"", ""Restaurants""]"
"[""Breakfast & Brunch"", ""American (New)"", ""Restaurants""]"
"[""Breakfast & Brunch"", ""American (New)"", ""Restaurants""]"


# Part 1

In [6]:
import re

def tokenize(text: str):
    return re.findall(r"[a-zA-Z']+", text.lower())

yelp = yelp.with_columns(
    pl.col("text").map_elements(tokenize, return_dtype = pl.List(pl.String)).alias("tokens")
)

rest_reviews = yelp.filter(
    (pl.col("name") == restaurant_name) &
    (pl.col("city") == restaurant_city)
)

In [7]:
from collections import Counter

def word_counts(df: pl.DataFrame, token_col: str = "tokens") -> Counter:
    c = Counter()
    for toks in df[token_col]:
        c.update(toks)
    return c

In [8]:
one_star = rest_reviews.filter(pl.col("stars") == 1)
five_star = rest_reviews.filter(pl.col("stars") == 5)

rest_not1 = rest_reviews.filter(pl.col("stars") != 1)
rest_not5 = rest_reviews.filter(pl.col("stars") != 5)

c1 = word_counts(one_star)
c5 = word_counts(five_star)
c_rest1 = word_counts(rest_not1)
c_rest5 = word_counts(rest_not5)

In [9]:
import math

def log_likelihood(k1, n1, k2, n2):
    if k1 == 0 and k2 == 0:
        return 0.0
    E1 = n1 * (k1 + k2) / (n1 + n2)
    E2 = n2 * (k1 + k2) / (n1 + n2)
    return 2 * (
        (k1 * math.log(k1 / E1) if k1 > 0 else 0.0) +
        (k2 * math.log(k2 / E2) if k2 > 0 else 0.0)
    )

def compute_keyness(c_target: Counter, c_ref: Counter):
    n1 = sum(c_target.values())
    n2 = sum(c_ref.values())
    vocab = set(c_target) | set(c_ref)
    rows = []
    for w in vocab:
        k1 = c_target.get(w, 0)
        k2 = c_ref.get(w, 0)
        score = log_likelihood(k1, n1, k2, n2)
        rows.append((w, score, k1, k2))
    rows.sort(key=lambda x: x[1], reverse=True)
    return rows

2. I chose a log-likelihood metric because it takes account for unbalanced datasets. Since there are typically more 4 and 5 star reviews, log-likelihood adjusts for this imbalance, so a word isn't unfairly inflated just because one group is larger.

In [10]:
key_1star = compute_keyness(c1, c_rest1)
key_5star = compute_keyness(c5, c_rest5)

In [11]:
import pandas as pd

df_1star = pd.DataFrame(key_1star, columns=["word", "keyness", "count_target", "count_ref"])
df_5star = pd.DataFrame(key_5star, columns=["word", "keyness", "count_target", "count_ref"])

display(df_1star.head(20))
display(df_5star.head(20))

,word,keyness,count_target,count_ref
0,worst,221.054163,69,37
1,manager,213.299134,95,116
2,asked,197.175595,148,367
3,cold,174.641008,87,128
4,portions,163.515493,70,2889
5,huge,149.022914,65,2658
6,horrible,135.406477,61,76
7,never,125.348552,145,525
8,fried,123.136942,62,2366
9,great,122.963308,70,2524


,word,keyness,count_target,count_ref
0,amazing,371.056846,625,443
1,great,263.897748,1172,1422
2,vegas,238.336827,991,1172
3,delicious,216.539244,660,680
4,best,191.377332,506,484
5,awesome,165.803509,344,284
6,love,159.010754,421,403
7,bland,142.289714,7,267
8,must,126.469432,294,261
9,dry,123.916077,33,386


In [12]:
competitors = yelp.filter(
    (pl.col("city") == restaurant_city) &
    (pl.col("name") != restaurant_name)
)

len(competitors)

198662

In [13]:
c_rest_all = word_counts(rest_reviews)
c_comp_all = word_counts(competitors)

key_vs_comp = compute_keyness(c_rest_all, c_comp_all)

df_vs_comp = pd.DataFrame(
    key_vs_comp, columns=["word", "keyness", "count_target", "count_ref"]
)

df_vs_comp_filtered = df_vs_comp[
    (df_vs_comp["count_target"] >= 10)
].head(30)

df_vs_comp_filtered

,word,keyness,count_target,count_ref
0,waffles,13955.501326,2801,1594
1,hash,12472.593688,2839,2627
2,sage,9054.064458,1448,172
3,portions,7673.456070,2959,9438
4,chicken,7064.222030,5365,39893
5,huge,5498.736118,2723,12607
6,benedict,4865.069290,1280,1827
7,biscuit,4033.333853,775,359
8,bacon,3625.265113,1880,9240
9,house,3559.408098,1912,9820


3. Strengths: The words "portion" and "huge" appear with extremely high distinctiveness. These words by itself may seem like it could have a positive or negative meaning, but together they have a positive correlation. Big portion sizes is this restaurants biggest strength compared to its other competitiors in Las Vegas. There are also many breakfast foods highly associated with Hash House A Go Go compared to its competitiors. This may seem like an obvious inference, but this gives this strong association with breakfast foods gives it competitive advantage over the breakfast/brunch market in Las Vegas, as people remeber Hash House A Go Go for its standout brunch foods, making it a top choice for people looking to get breakfast in Las Vegas

   Weaknesses: The word "wait" has a keyness of 2340 meaning it is very associated with Hash House A Go Go compared to its competitors. This restaurant expericnes many wait complaints which is a huge weakness for customer satisfaction. 

5. This competitor comparison revealed that Hash House A Go Go Las Vegas has a clear advantage over the breakfast/brunch market in Vegas. Many people pick this spot for their morning meal over other restaurants. A client should pay attention to two things: portion sizes and wait times. Portion sizes are this restaruants biggest strength, and they should keep huge portion sizes as one of their top priorties. The second thing a client should pay attention to is wait times. This is the restaurant's biggest weakness and must focus on shortening the time it takes to get seated and be served.

# Part 2

In [14]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 84.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [15]:
import spacy
nlp = spacy.load("en_core_web_sm")

rest_texts = rest_reviews["text"].to_list()

1.

In [16]:
def extract_aspects(text):
    doc = nlp(text)
    aspects = []

    for token in doc:

        # Pattern 1: adjective modifies noun → "huge portions"
        if token.dep_ == "amod" and token.head.pos_ == "NOUN":
            aspects.append((token.head.lemma_, token.lemma_))

        # Pattern 2: noun + copular verb + adjective → "service was slow"
        if token.dep_ == "acomp" and token.head.pos_ == "VERB":
            for child in token.head.children:
                if child.dep_ == "nsubj" and child.pos_ == "NOUN":
                    aspects.append((child.lemma_, token.lemma_))

        # Pattern 3: verb + direct object → "loved the waffles"
        if token.dep_ == "dobj" and token.head.pos_ == "VERB":
            aspects.append((token.lemma_, token.head.lemma_))

    return aspects

In [17]:
all_aspects = []

for review in rest_texts:
    pairs = extract_aspects(review)
    all_aspects.extend(pairs)

In [18]:
import pandas as pd

df_aspects = pd.DataFrame(all_aspects, columns=["aspect", "opinion"])
df_aspects.head(20)

,aspect,opinion
0,location,second
1,choice,favorite
2,thing,do
3,ingredient,fresh
4,ingredient,describe
5,portion,large
6,charge,avoid
7,person,other
8,side,order
9,2.50,charge


In [19]:
aspect_counts = df_aspects.groupby("aspect").size().reset_index(name="count")
aspect_counts.sort_values("count", ascending=False).head(20)

,aspect,count
2160,it,2705
1746,food,2700
1097,chicken,2029
2806,portion,1909
2768,place,1690
3544,time,1642
3522,thing,1094
2396,meal,967
2818,potato,945
2775,plate,880


In [20]:
aspect_opinion_counts = (
    df_aspects.groupby(["aspect", "opinion"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

aspect_opinion_counts.head(30)

,aspect,opinion,count
16932,portion,huge,579
17191,potato,mashed,512
4750,chicken,fry,428
4749,chicken,fried,387
22710,time,next,325
22868,toast,french,299
9023,food,good,262
16949,portion,large,260
22668,time,first,234
9028,food,great,226


2.

In [21]:
positive_words = {
    "amazing", "great", "good", "delicious", "friendly", "perfect",
    "crispy", "fluffy", "tasty", "fresh", "awesome", "fantastic"
}

negative_words = {
    "slow", "cold", "dry", "bland", "salty", "overcooked",
    "long", "rude", "bad", "terrible", "awful"
}

def opinion_score(opinion):
    if opinion in positive_words:
        return 1
    if opinion in negative_words:
        return -1
    return 0

In [22]:
review_scores = []

for i, review in enumerate(rest_texts):
    pairs = extract_aspects(review)
    score = sum(opinion_score(op) for _, op in pairs)
    review_scores.append(score)

In [23]:
df_reviews = rest_reviews.to_pandas()
df_reviews["orientation_score"] = review_scores

In [24]:
df_reviews[["stars", "orientation_score"]].corr()

,stars,orientation_score
stars,1.000000,0.234311
orientation_score,0.234311,1.000000


In [25]:
df_reviews.groupby("stars")["orientation_score"].mean()

stars
1   -0.203518
2    0.205455
3    0.437086
4    0.673040
5    0.778503
Name: orientation_score, dtype: float64

2. Orientation scores predict star ratings reasonably well. The correlation between the two is 0.23, which shows a clear positive relationship even if it’s not extremely strong. The pattern by star level is much stronger: average orientation scores rise steadily from –0.20 for 1‑star reviews to +0.78 for 5‑star reviews. That smooth increase shows that reviews with more positive aspect–opinion pairs consistently receive higher star ratings, so orientation scores capture the overall sentiment of the review in a meaningful and predictable way.


In [26]:
def classify_sentiment(opinion):
    if opinion in positive_words:
        return "positive"
    if opinion in negative_words:
        return "negative"
    return "neutral"

In [27]:
df_aspects["sentiment"] = df_aspects["opinion"].apply(classify_sentiment)

In [28]:
df_aspects[df_aspects["sentiment"] == "positive"] \
    .groupby("aspect").size().sort_values(ascending=False).head(10)

aspect
food          708
service       339
place         291
thing         171
breakfast     148
experience    114
meal           99
portion        94
price          76
spinach        66
dtype: int64

In [29]:
df_aspects[df_aspects["sentiment"] == "negative"] \
    .groupby("aspect").size().sort_values(ascending=False).head(10)

aspect
wait          130
time          105
service        81
food           52
line           44
tomato         37
thing          32
experience     32
night          32
side           28
dtype: int64

In [30]:
[x for x in rest_texts if "service" in x.lower()][:10]

['Five stars for a brunch place? Hell yeah!  (I haven\'t had their dinner menu, but for a weekend brunch, it hits the spot!) \n\nAfter driving by numerous times and being intrigued by the name of the restaurant alone, it was about time I got a chance to experience what the Hash House was really all about.  \n\nAfter about a 20 minute wait, we were starving.  It didn\'t help that we could see (stare) at what the diners were chowing on as they sat on the patio.  Gigantic portions of anything and everything, that\'s for sure!\n\nI had their special Mimosa with Pom Tangerine.  Can I say yum?  If my boyfriend will make me a pitcher of that, I promise to get drunk real fast!  My other dining companions had their HH Bloody Mary with all the fixings.  Those darn fixings were fabuloso!  They came with these vinegared Blue Lake string beans and stuffed green olives.  I heard two exclamations of "I think this the BEST Bloody Mary I have ever had!"  Hot damn!  That was 2 for 2!\n\nFor food, I orde

3. The most frequent positive aspects in the reviews are food, service, breakfast, experience and portion size. These aspects are consistently paired with positive opinion words such as “huge,” “delicious,” “crispy,” and “friendly”. The most frequent negative aspects are service speed, wait times, service and lines. To validate these findings, I spot‑checked the aspect “service,” which appears frequently in both positive and negative contexts. Positive mentions describe the service as friendly and attentive, while negative mentions highlight slow or inconsistent service. This confirms that service is a mixed aspect and that the aspect–opinion extraction accurately reflects customer sentiment.


4.

In [31]:
import random
for r in random.sample(rest_texts, 5):
    print(r, "\n")

Absolutely amazing food! Not too big of a menu which is good, but what they've got is not gonna disappoint. Was in Vegas to celebrate my friend's Bday and he and his wife/my bff, wanted to hit up this spot cuz they love it. I almost passed on it but I'm glad I didn't. The portions are humungous! I loved every bit of my chicken n waffles. The drinks were beyond good. Service was great, it was a theme day so all the servers were dressed up like robots, which was cool. DJ was playing some good music. I find myself day dreaming about this place still 4 days later. I definitely WILL be going back the next time I hit Vegas! They need to open one in LA!!! 

Was thoroughly disappointed...  I was so impressed last I went, I took my brothers and father there to showoff the good food.  Apparently it's best to go in the morning or afternoon...  We went know after work at about 5pm.. Ordered two mushroom Swiss burgers and two fried chicken Eggs Benedict (their man VS food) the cooks today just didn

This is one example of where the system can go wrong. As the word "service" is mentioned, followed by the words "great" and" "horrible". This could throw off the system and possibly classify it as a positve and/or negative aspect, even though the service was mediocore.

This is another example of where the system may incorrectly classify an aspect. The word "sucked" is mentioned followed by the word "breakfast". The system may classify breakfast as a negative aspect becase of the word "sucked", even though they are talking about the hotel.

I would not fully trust this tool without human intervention. This is because the system is useful for summarizing broad patterns, but it has predictable weaknesses such as sarcasm and rhetorical language. The system is reliable enough for exploratory analysis, but not for automated decision‑making without human review.

# Part 3

In [32]:
from absa import absa

Device set to use cuda:0
/home/jovyan/shared/ling-583/text_mining/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Device set to use cuda:0


In [33]:
import polars as pl

df_reviews = pl.DataFrame({"text": rest_texts})

In [34]:
df_absa = absa(df_reviews)

Extracting :   0%|          | 0/56841 [00:00<?, ?it/s]

Classifying:   0%|          | 0/66153 [00:00<?, ?it/s]

In [35]:
df_absa.head(10)

text,sentence,aspect,score
str,str,str,f64
"""Second location I've been to, …","""Second location I've been to, …","""breakfast""",0.996852
"""Second location I've been to, …","""Our server was most helpful, s…","""server""",0.99233
"""Second location I've been to, …","""Our server was most helpful, s…","""ingredients""",0.98822
"""Second location I've been to, …","""Our server was most helpful, s…","""portions""",0.988649
"""Second location I've been to, …","""The flapjcks can be split betw…","""pizza""",0.0
"""Second location I've been to, …","""The smoked salmon hash is the …","""smoked salmon hash""",0.936303
"""Second location I've been to, …","""Their house made strawberry ja…","""strawberry jam""",0.996173
"""Second location I've been to, …","""Their house made strawberry ja…","""biscuits""",0.0
"""Second location I've been to, …","""Its larger than the San Diego …","""wait""",-0.5426


In [36]:
df_absa["aspect"].value_counts().sort("count", descending=True).head(10)

aspect,count
str,u32
"""food""",5292
"""portions""",2470
"""place""",2299
"""service""",1597
"""wait""",1581
"""breakfast""",1414
"""chicken""",1151
"""meal""",992
"""Hash House""",899


In [37]:
yelp_hh_lv = yelp.filter(
    (pl.col("name") == "Hash House A Go Go") &
    (pl.col("city") == "Las Vegas")
)

In [38]:
df_absa = df_absa.drop(["stars", "stars_right", "stars_left"], strict=False)

df_absa = df_absa.join(
    yelp_hh_lv.select(["text", "stars"]),
    on="text",
    how="left"
)

In [39]:
df_absa.group_by("stars").agg(
    pl.col("score").mean().alias("avg_sentiment")
).sort("stars")

stars,avg_sentiment
i64,f64
1,-0.29335
2,-0.112726
3,0.109081
4,0.340264
5,0.471983


In [40]:
df_absa = df_absa.join(
    yelp_hh_lv.select(["text", "stars"]),
    on="text",
    how="left"
)

In [41]:
# 1-Star Reviews

display(df_absa.filter(pl.col("stars") == 1)["aspect"].value_counts().sort("count", descending = True).head(10))

aspect,count
str,u32
"""food""",445
"""place""",159
"""service""",145
"""server""",103
"""breakfast""",84
"""waitress""",83
"""waiter""",79
"""table""",75
"""eggs""",74


In [42]:
# 5-Star Reviews

display(df_absa.filter(pl.col("stars") == 5)["aspect"].value_counts().sort("count", descending = True).head(10))

aspect,count
str,u32
"""food""",1686
"""place""",926
"""portions""",892
"""wait""",592
"""service""",554
"""breakfast""",551
"""chicken""",350
"""meal""",325
"""Hash House""",322


1. Frequent Aspects: This distribution tells the client that diners consistently focus on food quality, portion size, and the overall dining experience, with service and wait times also playing a major role in how reviews are framed.
   
   Sentiment by Star Rating: The constant increase shows that the model is capturing real emotional differences across star ratings, not noise. It also validates that the aspect‑level sentiment scores align with how customers rate their experience overall.

   1-Star Reviews: These aspects point to operational breakdowns such as service interactions, table management, and inconsistent food execution. Even though “food” is the top aspect, the sentiment attached to it in 1‑star reviews is overall negative.

   5-Star Reviews: The same themes appear, but with positive sentiment. This symmetry with the same aspects but opposite sentiments illustrates that the restaurnt's strengths and weaknesses are tightly linked to the same core elements of the experience.

   Client Takeaways:
   1. Food is the heart of the restaurant. Customers talk more about food than anything else and positive reviews consitently highlight portions, breakfast items, chicken and waffles.
   2. Operational issues drive negative reviews. The negative side is dominated by service interactions, wait times, and inconsistent food execution. These are solvable operational problems such as staffing, training, and kitchen consistency, not fundamental brand weaknesses.
   3. The same aspects drive both praise and complaints. This is a powerful insight as customers care deeply about the same things in both good and bad experiences. Improving service reliability and food consistency would directly reduce 1‑star reviews without changing what people already love.
   4. The model output's are trustworthy. The sentiment scores align perfectly with star ratings and the transformer extracts clean, specific aspects instead of noise. The patterns match what humans expect, bad service = low stars and great food = high stars. This combination of volume, consistency, and model quality makes the insights reliable enough to guide real operational decisions.


In [43]:
df_parser = pd.concat([
    df_1star.assign(stars=1),
    df_5star.assign(stars=5)
])

In [44]:
import polars as pl

df_parser = pl.from_pandas(df_parser)

In [45]:
df_parser.columns

['word', 'keyness', 'count_target', 'count_ref', 'stars']

In [46]:
df_parser = df_parser.with_columns(
    pl.when(pl.col("stars") == 5).then(pl.lit("positive"))
     .when(pl.col("stars") == 1).then(pl.lit("negative"))
     .otherwise(pl.lit("neutral"))
     .alias("parser_polarity")
)

In [47]:
df_parser = df_parser.with_columns(
    pl.col("stars")
      .map_elements(
          lambda s: "positive" if s == 5
          else "negative" if s == 1
          else "neutral"
      )
      .alias("parser_polarity")
)

In [48]:
print(df_parser.columns)
print(df_parser.head())

['word', 'keyness', 'count_target', 'count_ref', 'stars', 'parser_polarity']
shape: (5, 6)
┌──────────┬────────────┬──────────────┬───────────┬───────┬─────────────────┐
│ word     ┆ keyness    ┆ count_target ┆ count_ref ┆ stars ┆ parser_polarity │
│ ---      ┆ ---        ┆ ---          ┆ ---       ┆ ---   ┆ ---             │
│ str      ┆ f64        ┆ i64          ┆ i64       ┆ i64   ┆ str             │
╞══════════╪════════════╪══════════════╪═══════════╪═══════╪═════════════════╡
│ worst    ┆ 221.054163 ┆ 69           ┆ 37        ┆ 1     ┆ negative        │
│ manager  ┆ 213.299134 ┆ 95           ┆ 116       ┆ 1     ┆ negative        │
│ asked    ┆ 197.175595 ┆ 148          ┆ 367       ┆ 1     ┆ negative        │
│ cold     ┆ 174.641008 ┆ 87           ┆ 128       ┆ 1     ┆ negative        │
│ portions ┆ 163.515493 ┆ 70           ┆ 2889      ┆ 1     ┆ negative        │
└──────────┴────────────┴──────────────┴───────────┴───────┴─────────────────┘


In [49]:
df_parser = df_parser.rename({"word": "aspect"})

In [50]:
df_absa = df_absa.drop(pl.col("^stars.*$"), strict=False)

df_absa = df_absa.with_columns(
    pl.when(pl.col("score") > 0).then(pl.lit("positive"))
     .when(pl.col("score") < 0).then(pl.lit("negative"))
     .otherwise(pl.lit("neutral"))
     .alias("transformer_polarity")
)

df_absa.columns

['text', 'sentence', 'aspect', 'score', 'transformer_polarity']

In [51]:
df_compare = df_parser.join(
    df_absa,
    on="aspect",
    how="inner"
)

In [52]:
df_disagree = df_compare.filter(
    pl.col("parser_polarity") != pl.col("transformer_polarity")
)

In [53]:
df_top3 = df_disagree.sort(
    pl.col("score").abs(),
    descending=True
).head(3)

df_top3.select([
    "text",
    "aspect",
    "parser_polarity",
    "transformer_polarity",
    "score",
    "stars"
])

text,aspect,parser_polarity,transformer_polarity,score,stars
str,str,str,str,f64,i64
"""i really love this restaurant …","""restaurant""","""negative""","""positive""",0.998897,1
"""Big portions but, food was kin…","""hostess""","""negative""","""positive""",0.998894,1
"""Great atmosphere. The servers …","""servers""","""negative""","""positive""",0.998833,1


In [54]:
df_top3.select("text").to_series().to_list()

['i really love this restaurant it very original ... i think it a very unique looking place and serves very great portions of food my bf and his dad order biscuits and gravy with eggs and bacon and i order the pancake..which was bigger then the plate.. my bf mom order a standard breakfast but they all look big and delicious and well made ..the prices were awesome .. the food taste really good so it was a good experience nome!!! not your run of the mill breakfast and brunch plate..it was like a homemade southern breakfast plate more of an old style which a high end plating.. which made it look fancy  lol t too oh they played a lot of johnny cash which tie everything together.. lol i love this place and will be going back!!!',
 'Big portions but, food was kinda bland!!! the hostess are excellent... Our waitress (Ingrid) was HORRIBLE!!! She had a very poor attitude! She came to our table twice, she never brought us napkins nor did she ever refill our water glasses! We asked for a side of 

2. In all three reviews, the parser classified the aspect as negative, but the transformer classified it as positive.
   
   Review #1: This review should be classifed as a positive as the review only talks about good things about the restaurant, and even talks about going there again.

   Review #2: This review is tehnically correct in terms of the aspect of "hostess", but the review as a whole is very negative and continously lists things that negatively affected their expereince.

   Reveiew #3: This review should be identified as positive, as the review only mentions good things about their server.

3. The transformer‑based ABSA model is the better long‑term investment because it consistently delivers accurate, actionable insights that the parser method simply cannot match. In the results, the parser mislabeled clearly positive sentences as negative just because they appeared in 1‑star reviews, and it frequently attached sentiment to the wrong aspect (“restaurant,” “hostess,” “servers”), while the transformer correctly interpreted sentence‑level meaning, contrastive structures, and implicit sentiment. This accuracy matters because it leads to precise operational identifying issues like slow seating or inconsistent service rather than just surfacing frequent words. The transformer does cost more to run, but the reduction in false insights and the improvement in decision quality more than justify the expense. The simpler parser method is still useful when the client only needs high‑level trend monitoring or extremely low‑cost, large‑scale keyword tracking, but for any analysis that informs real business decisions, the transformer model is the more reliable choice.

4. Fully‑automated text analysis is helpful for getting a quick, big‑picture view of what customers talk about, but it shouldn’t be treated as a perfect replacement for human judgment. The models do a good job spotting common themes like service, wait times, or food quality, and they can process thousands of reviews much faster than a person ever could. But they still make mistakes, especially with mixed or sarcastic reviews, or when a sentence has both praise and criticism. Because of that, the client should trust the automated system for broad trends and early signals, but a human should still check surprising or important findings before the business acts on them. Trying to automate every tiny detail isn’t worth the effort; the value comes from using the model to narrow things down, then letting a person confirm the parts that matter most.

# Part 4

In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import seaborn as sns
import matplotlib.pyplot as plt
import polars as pl

# --- Extract lists from yelp ---
restaurant_list = (
    yelp.select("name")
        .unique()
        .get_column("name")
        .to_list()
)

city_list = (
    yelp.select("city")
        .unique()
        .get_column("city")
        .to_list()
)

# categories is a list column → flatten it
category_list = (
    yelp.select(pl.col("categories").list.explode())
        .unique()
        .get_column("categories")
        .to_list()
)

# --- Widgets ---
restaurant_dropdown = widgets.Dropdown(
    options=sorted(restaurant_list),
    description="Restaurant:",
    layout=widgets.Layout(width="350px")
)

compare_dropdown = widgets.Dropdown(
    options=["None"] + sorted(restaurant_list),
    description="Compare:",
    layout=widgets.Layout(width="350px")
)

city_filter = widgets.Dropdown(
    options=["All"] + sorted(city_list),
    description="City:",
    layout=widgets.Layout(width="200px")
)

category_filter = widgets.Dropdown(
    options=["All"] + sorted(category_list),
    description="Category:",
    layout=widgets.Layout(width="250px")
)

star_filter = widgets.Dropdown(
    options=["All", 1, 2, 3, 4, 5],
    description="Stars:",
    layout=widgets.Layout(width="150px")
)

output = widgets.Output()

# --- Update function ---
def update_view(*args):
    with output:
        clear_output()

        restaurant = restaurant_dropdown.value
        compare_to = compare_dropdown.value
        stars = star_filter.value
        city = city_filter.value
        category = category_filter.value

        if restaurant is None:
            return

        # Base filter for main restaurant
        data = yelp.filter(pl.col("name") == restaurant)

        # Apply city filter
        if city != "All":
            data = data.filter(pl.col("city") == city)

        # Apply category filter
        if category != "All":
            data = data.filter(pl.col("categories").list.contains(category))

        # Apply star filter
        if stars != "All":
            data = data.filter(pl.col("stars") == stars)

        # Comparison logic
        if compare_to != "None":
            data_comp = yelp.filter(pl.col("name") == compare_to)

            if city != "All":
                data_comp = data_comp.filter(pl.col("city") == city)

            if category != "All":
                data_comp = data_comp.filter(pl.col("categories").list.contains(category))

            if stars != "All":
                data_comp = data_comp.filter(pl.col("stars") == stars)

            fig, ax = plt.subplots(1, 2, figsize=(12, 4))

            sns.countplot(x="stars", data=data.to_pandas(), ax=ax[0])
            sns.countplot(x="stars", data=data_comp.to_pandas(), ax=ax[1])

            ax[0].set_title(f"{restaurant} — Star Distribution")
            ax[1].set_title(f"{compare_to} — Star Distribution")

            plt.show()

        else:
            # Single-restaurant view
            sns.countplot(x="stars", data=data.to_pandas())
            plt.title(f"{restaurant} — Star Distribution")
            plt.show()

# --- Attach listeners ---
restaurant_dropdown.observe(update_view, "value")
compare_dropdown.observe(update_view, "value")
city_filter.observe(update_view, "value")
category_filter.observe(update_view, "value")
star_filter.observe(update_view, "value")

# --- Layout ---
ui = widgets.VBox([
    widgets.HBox([restaurant_dropdown, compare_dropdown]),
    widgets.HBox([city_filter, category_filter, star_filter]),
    output
])

display(ui)
update_view()